# Data Collection for Sports World Model

This notebook collects play-by-play data and news articles to train our perplexity-based world model.

## Data Sources

1. **Play-by-Play Data**:
   - NBA Stats API (free, official)
   - BALLDONTLIE API (free tier available)
   - SportsDataIO (paid, very comprehensive)

2. **News/Context Data**:
   - SportsDataIO news endpoint
   - NewsAPI (general news)
   - Reddit r/nba (social sentiment)

3. **Betting Odds**:
   - The Odds API (free tier: 500 requests/month)
   - Historical odds for backtesting

## Goal

Collect 2-3 seasons of NBA data:
- ~1,230 games per season
- ~300-400 events per game
- ~400K-500K events per season
- Target: 1M+ events for training

In [ ]:
import sys
sys.path.append('../src')

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
from pathlib import Path
import time
from tqdm import tqdm

# Create data directories
Path('../data/raw/games').mkdir(parents=True, exist_ok=True)
Path('../data/raw/news').mkdir(parents=True, exist_ok=True)
Path('../data/raw/odds').mkdir(parents=True, exist_ok=True)

## 1. NBA Stats API - Play-by-Play Data

The NBA Stats API is free and provides detailed play-by-play data. No API key required!

In [ ]:
class NBAStatsCollector:
    """Collector for NBA Stats API data"""
    
    BASE_URL = "https://stats.nba.com/stats"
    
    HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json',
        'Referer': 'https://www.nba.com/'
    }
    
    def __init__(self, season='2024-25'):
        self.season = season
        
    def get_game_ids(self, start_date='2024-10-01', end_date='2025-04-30'):
        """
        Get all game IDs for a season.
        
        Returns:
            List of game IDs
        """
        endpoint = f"{self.BASE_URL}/leaguegamelog"
        params = {
            'Season': self.season,
            'SeasonType': 'Regular Season',
            'LeagueID': '00',
            'Direction': 'DESC',
            'Sorter': 'DATE'
        }
        
        response = requests.get(endpoint, headers=self.HEADERS, params=params)
        
        if response.status_code == 200:
            data = response.json()
            # Parse game IDs from response
            game_ids = []
            for row in data['resultSets'][0]['rowSet']:
                game_id = row[4]  # Game ID is typically 5th column
                if game_id not in game_ids:
                    game_ids.append(game_id)
            return game_ids
        else:
            print(f"Error: {response.status_code}")
            return []
    
    def get_play_by_play(self, game_id):
        """
        Get play-by-play data for a specific game.
        
        Args:
            game_id: NBA game ID (e.g., '0022400001')
            
        Returns:
            DataFrame with play-by-play events
        """
        endpoint = f"{self.BASE_URL}/playbyplayv3"
        params = {
            'GameID': game_id,
            'StartPeriod': 0,
            'EndPeriod': 10
        }
        
        response = requests.get(endpoint, headers=self.HEADERS, params=params)
        
        if response.status_code == 200:
            data = response.json()
            
            # Extract play-by-play data
            if 'resultSets' in data and len(data['resultSets']) > 0:
                headers = data['resultSets'][0]['headers']
                rows = data['resultSets'][0]['rowSet']
                df = pd.DataFrame(rows, columns=headers)
                return df
            else:
                return pd.DataFrame()
        else:
            print(f"Error fetching game {game_id}: {response.status_code}")
            return pd.DataFrame()
    
    def collect_season(self, max_games=None, delay=0.6):
        """
        Collect all play-by-play data for a season.
        
        Args:
            max_games: Maximum number of games to collect (None = all)
            delay: Delay between requests (seconds) to respect rate limits
        """
        print(f"Getting game IDs for {self.season}...")
        game_ids = self.get_game_ids()
        
        if max_games:
            game_ids = game_ids[:max_games]
        
        print(f"Found {len(game_ids)} games. Collecting play-by-play data...")
        
        for game_id in tqdm(game_ids):
            # Check if already downloaded
            save_path = f"../data/raw/games/{game_id}.parquet"
            if Path(save_path).exists():
                continue
            
            # Get play-by-play
            df = self.get_play_by_play(game_id)
            
            if not df.empty:
                # Save to parquet (efficient format)
                df.to_parquet(save_path, index=False)
            
            # Rate limiting
            time.sleep(delay)
        
        print(f"Collection complete! Data saved to ../data/raw/games/")

In [ ]:
# Example: Collect first 10 games
collector = NBAStatsCollector(season='2024-25')
collector.collect_season(max_games=10)

## 2. The Odds API - Futures Betting Lines

Collect historical and current futures odds for comparison with model predictions.

Sign up for free API key at: https://the-odds-api.com/
(500 requests/month on free tier)

In [ ]:
class OddsAPICollector:
    """Collector for betting odds data"""
    
    BASE_URL = "https://api.the-odds-api.com/v4"
    
    def __init__(self, api_key):
        self.api_key = api_key
    
    def get_futures_odds(self, sport='basketball_nba', market='outrights'):
        """
        Get futures odds (championship, playoff odds, etc.)
        
        Returns:
            DataFrame with futures odds
        """
        endpoint = f"{self.BASE_URL}/sports/{sport}/odds"
        params = {
            'apiKey': self.api_key,
            'regions': 'us',
            'markets': market,
            'oddsFormat': 'american'
        }
        
        response = requests.get(endpoint, params=params)
        
        if response.status_code == 200:
            data = response.json()
            # Parse odds data
            odds_records = []
            for event in data:
                for bookmaker in event.get('bookmakers', []):
                    for market in bookmaker.get('markets', []):
                        for outcome in market.get('outcomes', []):
                            odds_records.append({
                                'team': outcome['name'],
                                'odds': outcome['price'],
                                'bookmaker': bookmaker['title'],
                                'market': market['key'],
                                'timestamp': datetime.now().isoformat()
                            })
            return pd.DataFrame(odds_records)
        else:
            print(f"Error: {response.status_code}")
            return pd.DataFrame()
    
    def track_odds_movement(self, sport='basketball_nba', interval_hours=24):
        """
        Track odds movement over time (run this periodically)
        """
        df = self.get_futures_odds(sport)
        
        if not df.empty:
            # Append to historical log
            save_path = f"../data/raw/odds/{sport}_futures.csv"
            if Path(save_path).exists():
                df_existing = pd.read_csv(save_path)
                df = pd.concat([df_existing, df], ignore_index=True)
            
            df.to_csv(save_path, index=False)
            print(f"Odds data saved: {len(df)} records")
        
        return df

In [ ]:
# Example usage (need API key)
# API_KEY = "your_api_key_here"  # Get from https://the-odds-api.com/
# odds_collector = OddsAPICollector(API_KEY)
# futures_odds = odds_collector.track_odds_movement()

## 3. News Data Collection

Collect news articles about teams and players to incorporate as context in the model.

In [ ]:
class NewsCollector:
    """Collector for sports news articles"""
    
    def __init__(self, newsapi_key=None):
        self.newsapi_key = newsapi_key
    
    def search_news(self, query, from_date=None, to_date=None, max_articles=100):
        """
        Search for news articles using NewsAPI.
        
        Args:
            query: Search query (e.g., "Lakers" or "LeBron James")
            from_date: Start date (YYYY-MM-DD)
            to_date: End date (YYYY-MM-DD)
            max_articles: Maximum articles to retrieve
            
        Returns:
            DataFrame with news articles
        """
        if not self.newsapi_key:
            print("NewsAPI key required. Get one at: https://newsapi.org/")
            return pd.DataFrame()
        
        endpoint = "https://newsapi.org/v2/everything"
        params = {
            'apiKey': self.newsapi_key,
            'q': query + ' AND NBA',
            'language': 'en',
            'sortBy': 'publishedAt',
            'pageSize': min(max_articles, 100)
        }
        
        if from_date:
            params['from'] = from_date
        if to_date:
            params['to'] = to_date
        
        response = requests.get(endpoint, params=params)
        
        if response.status_code == 200:
            data = response.json()
            articles = data.get('articles', [])
            
            records = []
            for article in articles:
                records.append({
                    'title': article['title'],
                    'description': article['description'],
                    'content': article.get('content', ''),
                    'url': article['url'],
                    'published_at': article['publishedAt'],
                    'source': article['source']['name'],
                    'query': query
                })
            
            return pd.DataFrame(records)
        else:
            print(f"Error: {response.status_code}")
            return pd.DataFrame()
    
    def collect_team_news(self, teams, date_range_days=30):
        """
        Collect news for multiple teams.
        
        Args:
            teams: List of team names
            date_range_days: How many days back to search
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=date_range_days)
        
        all_articles = []
        
        for team in tqdm(teams):
            df = self.search_news(
                query=team,
                from_date=start_date.strftime('%Y-%m-%d'),
                to_date=end_date.strftime('%Y-%m-%d'),
                max_articles=50
            )
            all_articles.append(df)
            time.sleep(1)  # Rate limiting
        
        df_all = pd.concat(all_articles, ignore_index=True)
        
        # Save
        save_path = f"../data/raw/news/articles_{datetime.now().strftime('%Y%m%d')}.parquet"
        df_all.to_parquet(save_path, index=False)
        
        print(f"Collected {len(df_all)} articles")
        return df_all

## 4. Data Summary

Let's check what we've collected so far:

In [ ]:
def summarize_collected_data():
    """Summarize all collected data"""
    
    # Count games
    game_files = list(Path('../data/raw/games').glob('*.parquet'))
    print(f"Games collected: {len(game_files)}")
    
    if game_files:
        # Sample a game to show event count
        sample_game = pd.read_parquet(game_files[0])
        print(f"Events per game (sample): {len(sample_game)}")
        print(f"Estimated total events: {len(game_files) * len(sample_game):,}")
    
    # Count news articles
    news_files = list(Path('../data/raw/news').glob('*.parquet'))
    if news_files:
        total_articles = sum(len(pd.read_parquet(f)) for f in news_files)
        print(f"News articles collected: {total_articles:,}")
    
    # Check odds data
    odds_files = list(Path('../data/raw/odds').glob('*.csv'))
    if odds_files:
        total_odds = sum(len(pd.read_csv(f)) for f in odds_files)
        print(f"Odds records collected: {total_odds:,}")

summarize_collected_data()

## Next Steps

1. **Target**: Collect 2-3 full NBA seasons (~1M+ events)
2. **Next Notebook**: `02_event_tokenization.ipynb` - Convert raw data to token sequences
3. **Then**: `03_model_training.ipynb` - Train the world model to minimize perplexity

## Tips for Data Collection

- **Rate Limits**: Respect API rate limits (NBA Stats: ~1 req/sec, NewsAPI: ~500/day free)
- **Historical Data**: For backtesting, collect 2-3 seasons of past data
- **News Timing**: Match news timestamps to games for context injection
- **Storage**: Parquet format is 10x more efficient than CSV for large datasets

## Estimated Costs

- **NBA Stats API**: Free ✓
- **The Odds API**: Free tier (500 req/month) or $40/month for more
- **NewsAPI**: Free tier (500 req/day) or $449/month for production
- **Alternative**: SportsDataIO ($20/month) has play-by-play + news + odds in one API